# 01 — EMG representation decision (DEV USERS ONLY)

**Question:** should the main table use the 20 Hz envelope EMG (`env20`, 16 ch) or the 2 kHz-derived spectral features (`fs64`, 112 ch = 16 ch x 7 features)?

**Why dev users only.** The 28 fold users are the test set. Choosing a representation by looking at their accuracy would be selection on test data. This notebook uses only the 4 HPO/dev users (P005, P131, P116, P109) and their cached dev episodes (100 per user per cell). The 28-user classical results (notebook 02) are then *reported for both representations*, not used to choose.

**Decision rule (written 2026-10-29, before running this notebook on all dev episodes):**
- Adopt `fs64` for the main table if, at 10-way 1-shot, **EMG-only fs64 beats EMG-only env20 by >= 5 points on average and on at least 3 of 4 dev users**, under both classifiers, **and** balanced fused fs64+IMU is not below balanced fused env20+IMU.
- Otherwise keep `env20`, unless the env20 time-axis anomaly (FULL_RUNNING_LOG 2026-10-21 late) is confirmed. A confirmed anomaly rules env20 out regardless.
- Kai + PI make the final call. Either way, the paper reports the comparison.

**Caveats.**
- 4 users only, and trivial classifiers (no training).
- The fs64 standardisation constants were computed from these same 4 dev users. That gives a small advantage on exactly these users.
- "Balanced" fusion scales each modality block to equal total weight, so 72 IMU channels cannot dominate the distance by count alone.

In [1]:
import os, sys, pickle, json
from pathlib import Path
import numpy as np

REPO = Path.cwd()
while not (REPO / ".git").exists():
    REPO = REPO.parent
os.chdir(REPO)
for rel in ("system", "system/MAML"):
    sys.path.insert(0, str(REPO / rel))
os.environ.setdefault("CODE_DIR", str(REPO))
os.environ.setdefault("RUN_DIR", os.environ.get("TEMP", "/tmp"))
os.environ.setdefault("HPO_CONFIG", "legacy")

from episodes import episode_cache as ec
from episodes.episode_numpy import materialize
from MAML.maml_data_pipeline import reorient_tensor_dict

DS = REPO / "dataset" / "meta-learning-sup-que-ds"
def load_dict(name, emg_ch):
    d = pickle.load(open(DS / name, "rb"))
    return d, reorient_tensor_dict(d, {"emg_in_ch": emg_ch, "imu_in_ch": 72})   # -> (trials, C, T)


In [2]:
d20, td20 = load_dict("segfilt_rts_tensor_dict.pkl", 16)
d64, td64 = load_dict("segfilt_fs64_tensor_dict.pkl", 112)
es = ec.load("dev", None)
print("dev users:", es.users, "| cells:", es.cells)

def block(x):
    """flatten (n, C, T) and scale so the block has unit average squared norm per sample."""
    x = x.reshape(len(x), -1).astype(np.float64)
    return x / np.sqrt(x.shape[1])

def inputs(m, kind):
    if kind == "emg":  return block(m.emg_support), block(m.emg_query)
    if kind == "imu":  return block(m.imu_support), block(m.imu_query)
    return (np.hstack([block(m.emg_support), block(m.imu_support)]),
            np.hstack([block(m.emg_query), block(m.imu_query)]))

def classify(s, ys, q, n, rule):
    if rule == "proto_euclid":
        P = np.stack([s[ys == c].mean(0) for c in range(n)])
        return ((q[:, None] - P[None]) ** 2).sum(2).argmin(1)
    if rule == "1nn_l1":
        return ys[np.abs(q[:, None] - s[None]).sum(2).argmin(1)]
    raise ValueError(rule)

CONDS = [("env20", "emg"), ("fs64", "emg"), ("env20", "imu"), ("env20", "both"), ("fs64", "both")]
TD = {"env20": td20, "fs64": td64}

def run(cell, rule):
    out = {}
    for rep, kind in CONDS:
        per = {}
        for ep in es.select(*cell):
            m = materialize(ep, TD[rep])
            s, q = inputs(m, kind)
            per.setdefault(ep.user, []).append((classify(s, m.y_support, q, ep.n_way, rule) == m.y_query).mean())
        out[f"{rep}:{kind}"] = {u: float(np.mean(v)) for u, v in per.items()}
    return out

C:\Users\kdmen\anaconda3\envs\fl_torch\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.2.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


dev users: ('P005', 'P109', 'P116', 'P131') | cells: ((3, 1), (3, 3), (3, 5), (5, 1), (5, 3), (5, 5), (10, 1), (10, 3), (10, 5))


In [3]:
import pandas as pd
results = {}
for cell in [(10, 1), (10, 5), (5, 1), (3, 1)]:
    for rule in ("proto_euclid", "1nn_l1"):
        r = run(cell, rule)
        df = pd.DataFrame(r).T * 100
        df["mean"] = df.mean(1)
        results[(cell, rule)] = df
        print(f"\n=== {cell[0]}-way {cell[1]}-shot, {rule} (accuracy %, per dev user) ===")
        print(df.round(1).to_string())


=== 10-way 1-shot, proto_euclid (accuracy %, per dev user) ===
            P005  P109  P116  P131  mean
env20:emg   40.6  48.5  78.0  27.1  48.6
fs64:emg    77.1  82.5  94.0  56.1  77.4
env20:imu   34.2  47.8  74.9  36.9  48.5
env20:both  45.2  52.7  84.1  39.5  55.4
fs64:both   54.0  68.8  90.9  50.9  66.1



=== 10-way 1-shot, 1nn_l1 (accuracy %, per dev user) ===
            P005  P109  P116  P131  mean
env20:emg   53.5  64.1  88.3  37.0  60.7
fs64:emg    76.8  84.9  94.5  55.4  77.9
env20:imu   43.3  55.9  84.7  48.9  58.2
env20:both  52.9  65.4  92.1  52.0  65.6
fs64:both   71.2  84.8  95.4  60.8  78.0



=== 10-way 5-shot, proto_euclid (accuracy %, per dev user) ===
            P005  P109  P116  P131  mean
env20:emg   53.5  64.6  90.9  40.1  62.3
fs64:emg    92.8  94.4  97.8  83.5  92.1
env20:imu   50.4  66.5  90.2  58.5  66.4
env20:both  59.7  71.9  93.2  58.9  70.9
fs64:both   73.7  88.6  97.6  77.7  84.4



=== 10-way 5-shot, 1nn_l1 (accuracy %, per dev user) ===
            P005  P109  P116  P131  mean
env20:emg   70.6  89.0  96.1  49.9  76.4
fs64:emg    93.9  97.4  97.8  81.3  92.6
env20:imu   64.3  79.5  96.7  65.0  76.4
env20:both  72.9  89.7  97.3  71.2  82.8
fs64:both   90.5  97.8  97.8  84.0  92.5



=== 5-way 1-shot, proto_euclid (accuracy %, per dev user) ===
            P005  P109  P116  P131  mean
env20:emg   54.9  57.9  84.7  39.0  59.1
fs64:emg    84.5  88.5  95.1  69.7  84.4
env20:imu   42.9  56.8  80.0  47.9  56.9
env20:both  55.9  60.8  87.2  49.0  63.2
fs64:both   64.1  78.8  93.0  61.1  74.2



=== 5-way 1-shot, 1nn_l1 (accuracy %, per dev user) ===
            P005  P109  P116  P131  mean
env20:emg   67.8  73.5  92.1  51.5  71.2
fs64:emg    84.8  90.4  95.4  68.7  84.8
env20:imu   51.9  66.6  89.4  59.8  66.9
env20:both  63.1  75.4  94.3  63.0  74.0
fs64:both   79.2  90.1  96.0  72.0  84.3



=== 3-way 1-shot, proto_euclid (accuracy %, per dev user) ===
            P005  P109  P116  P131  mean
env20:emg   68.8  68.5  91.2  52.5  70.2
fs64:emg    91.3  93.9  98.5  79.4  90.8
env20:imu   57.4  66.2  87.9  59.1  67.7
env20:both  68.3  70.7  92.9  60.5  73.1
fs64:both   74.7  86.0  97.2  72.0  82.5



=== 3-way 1-shot, 1nn_l1 (accuracy %, per dev user) ===
            P005  P109  P116  P131  mean
env20:emg   80.7  82.9  95.6  64.9  81.0
fs64:emg    90.5  94.3  98.3  77.5  90.1
env20:imu   65.1  75.2  94.1  72.2  76.6
env20:both  75.6  83.6  97.4  74.4  82.7
fs64:both   86.8  94.0  98.7  81.8  90.3


In [4]:
# Apply the pre-registered rule at 10-way 1-shot
ok = True
for rule in ("proto_euclid", "1nn_l1"):
    df = results[((10, 1), rule)]
    users = [c for c in df.columns if c != "mean"]
    d = df.loc["fs64:emg", users] - df.loc["env20:emg", users]
    fused = df.loc["fs64:both", "mean"] - df.loc["env20:both", "mean"]
    cond = d.mean() >= 5 and (d > 0).sum() >= 3 and fused >= 0
    ok &= cond
    print(f"{rule}: EMG-only fs64-env20 = {d.mean():+.1f} pts (users better: {(d > 0).sum()}/4); "
          f"fused fs64-env20 = {fused:+.1f} pts -> {'PASS' if cond else 'FAIL'}")
print("\nRule says:", "ADOPT fs64" if ok else "do NOT adopt fs64 on this evidence (see the anomaly clause)")

proto_euclid: EMG-only fs64-env20 = +28.9 pts (users better: 4/4); fused fs64-env20 = +10.8 pts -> PASS
1nn_l1: EMG-only fs64-env20 = +17.2 pts (users better: 4/4); fused fs64-env20 = +12.5 pts -> PASS

Rule says: ADOPT fs64
